<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Agent/blob/main/07%20Deep%20Agent%20Workflow/1_Alpinion_DeepAgents_Regulatory_Workflow_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 알피니언메디칼시스템 PBL 실습 교안
## LangChain Deep Agents로 Product-A 인허가 제출 준비 검토

**대상**: 프로그래밍 경험이 없는 수강생 · **권장 시간**: 180분

**실습 질문**: 제출 준비를 막고 있는 문서는 무엇이며, 누가 어떤 개정본을 확인해야 할까요?

- **실행**: Colab에 이 파일을 업로드하고 **런타임 → 모두 실행**을 선택합니다. 위에서 아래로 셀을 실행해도 됩니다.
- **기본 모드**: 가상 카탈로그와 Python 함수를 실행합니다. API 키가 없어도 문서 조회·준비 판정·응답 예시·보완표·검증을 실습할 수 있습니다.
- **실제 Agent 모드**: 3단계에서 API 키를 등록하고 스위치를 켭니다. 모델이 도구를 고르고 답변을 작성하는 모습을 확인합니다.
- **준비물**: 가상 데이터와 간단한 Traceability Matrix 예시는 노트북에 포함되어 있습니다.

업무 자료는 가상이며, 승인 상태 변경은 교육용 시나리오입니다. RA가 제출본을 확정하고 기존 결재 절차에 따라 기관 제출을 진행합니다.

| 용어 | 이 실습에서의 의미 |
|---|---|
| Tool | 문서나 시험 근거를 조회하는 Python 함수 |
| Deep Agent | 질문을 읽고 필요한 도구를 호출해 결과를 설명하는 업무 보조자 |
| 검증 규칙 | 필수 목록과 상태를 대조해 제출 준비 여부를 계산하는 Python 코드 |

## 1. 문서 카탈로그 읽기 · 20분
**따라 하기**: 다음 셀들을 실행합니다. 제출 필수 문서 8건과, 두 출처에서 수집한 개정본 목록을 확인합니다.  
**읽는 법**: 승인 상태는 문서의 검토 결과입니다. 제출 지정은 이번 프로젝트에서 어떤 개정본을 사용할지 정한 값입니다.  
**확인**: QVR-501의 v1.0과 v1.1을 찾아 두 상태를 비교해 보세요.

In [1]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory
from threading import Thread
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import urlopen
from zipfile import ZipFile

import pandas as pd
from IPython.display import display

PROJECT, PRODUCT, MARKET = "PA-001", "Product-A", "시장 A"
workflow = pd.DataFrame([
    ("A", "개발", "설계·시험 산출물 작성", "—"),
    ("B", "개발·RA", "제출 대상 판별 및 지정", "A"),
    ("C1", "품질", "품질·인증 시험 근거 준비", "B"),
    ("C2", "개발", "임상평가 자료 준비", "B"),
    ("C3", "구매·제조", "공급업체·제조 자료 준비", "B"),
    ("D", "RA", "STED 취합 및 완료", "C1, C2, C3"),
    ("E", "RA·승인권자", "결재 후 기관 제출", "D"),
    ("F", "RA·관련 부서", "기관 요청 시 보완 및 재제출", "E, 기관 보완 요청"),
], columns=["단계", "담당", "활동", "선행 조건"])
required = pd.DataFrame([
    ("DHF-001", "시스템 설계명세", "개발", "설계", "B"),
    ("QVR-501", "거리 측정 품질검증 보고서", "품질", "시험·검증", "D"),
    ("CER-001", "임상평가 보고서", "개발", "임상평가", "D"),
    ("SUP-001", "협력업체 승인원", "구매", "제조·공급망", "D"),
    ("DMR-001", "제조 기준 문서", "제조", "제조·공급망", "D"),
    ("STED-001", "기술문서 제출본", "RA", "인허가 제출·보완", "E"),
    ("TEST-001", "인증시험 성적서", "품질", "시험·검증", "E"),
    ("QVR-102", "영상 기능 품질검증 보고서", "품질", "시험·검증", "D"),
], columns=["document_id", "필수 문서", "책임부서", "예상 분류", "영향 단계"])
display(workflow, required)

,단계,담당,활동,선행 조건
0,A,개발,설계·시험 산출물 작성,—
1,B,개발·RA,제출 대상 판별 및 지정,A
2,C1,품질,품질·인증 시험 근거 준비,B
3,C2,개발,임상평가 자료 준비,B
4,C3,구매·제조,공급업체·제조 자료 준비,B
5,D,RA,STED 취합 및 완료,"C1, C2, C3"
6,E,RA·승인권자,결재 후 기관 제출,D
7,F,RA·관련 부서,기관 요청 시 보완 및 재제출,"E, 기관 보완 요청"


,document_id,필수 문서,책임부서,예상 분류,영향 단계
0,DHF-001,시스템 설계명세,개발,설계,B
1,QVR-501,거리 측정 품질검증 보고서,품질,시험·검증,D
2,CER-001,임상평가 보고서,개발,임상평가,D
3,SUP-001,협력업체 승인원,구매,제조·공급망,D
4,DMR-001,제조 기준 문서,제조,제조·공급망,D
5,STED-001,기술문서 제출본,RA,인허가 제출·보완,E
6,TEST-001,인증시험 성적서,품질,시험·검증,E
7,QVR-102,영상 기능 품질검증 보고서,품질,시험·검증,D


아래 준비 셀은 기존 1일차 실습의 폴더·읽기 전용 모의 API 자료를 불러오고, 동일 개정본의 출처를 합칩니다. 처음에는 코드를 수정하지 않고 실행하세요.

In [2]:
def record(doc_id, revision, title, kind, approval, designated=False,
           access="접근 가능", related=()):
    return dict(document_id=doc_id, revision=revision, title=title,
                source_type=kind, approval_status=approval,
                submission_status="지정" if designated else "미지정",
                access_status=access, product_id=PRODUCT, project_id=PROJECT,
                target_market=MARKET, regulation_refs=[],
                regulation_status="판본 확인 필요", related_ids=list(related))

folder_rows = [
    record("DHF-001", "1.0", "시스템 설계명세", "DESIGN", "승인", True),
    record("QVR-501", "1.0", "거리 측정 품질검증 보고서", "QVR", "승인",
           related=("EX-SRS-501", "EX-QVS-501")),
    record("QVR-501", "1.1", "거리 측정 품질검증 개정본", "QVR", "검토 중", True,
           related=("EX-SRS-501", "EX-QVS-501")),
    record("CER-001", "1.0", "임상평가 보고서", "CER", "승인", True),
    record("CER-001", "1.1", "임상평가 개정 초안", "CER", "검토 중"),
    record("STED-001", "0.2", "STED 취합 초안", "STED", "검토 중", True),
]
api_seed = [
    folder_rows[1].copy(),
    record("DMR-001", "1.0", "제조 기준 문서", "DMR", "승인", True, "접근 제한"),
    record("TEST-001", "2.0", "인증시험 성적서", "TEST", "승인"),
    record("QVR-102", "1.9", "영상 기능 품질검증 보고서", "QVR", "승인", True),
    record("QVR-102", "1.10", "영상 기능 품질검증 개정본", "QVR", "검토 중"),
    record("CR-001", "1.0", "거리 측정 시험 조건 변경 기록", "CHANGE", "승인",
           related=("EX-SRS-501",)),
]

def source_key(row):
    return f"{row['document_id']}@{row['revision']}"

with TemporaryDirectory() as workspace:
    project_folder = Path(workspace) / "Product-A"
    project_folder.mkdir()
    for row in folder_rows:
        file = project_folder / f"{source_key(row)}.json"
        file.write_text(json.dumps(row, ensure_ascii=False), encoding="utf-8")
    from_folder = []
    for file in sorted(project_folder.glob("*.json")):
        row = json.loads(file.read_text(encoding="utf-8"))
        from_folder.append(dict(row, source_id=file.stem, source_ref=f"폴더 Product-A/{file.name}"))

class CatalogHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path != "/documents":
            self.send_error(404)
            return
        body = json.dumps({"documents": api_seed}, ensure_ascii=False).encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *_):
        pass

with ThreadingHTTPServer(("127.0.0.1", 0), CatalogHandler) as server:
    thread = Thread(target=server.serve_forever, daemon=True)
    thread.start()
    try:
        with urlopen(f"http://127.0.0.1:{server.server_port}/documents", timeout=10) as response:
            api_rows = json.load(response)["documents"]
    finally:
        server.shutdown()
        thread.join()

from_api = [
    dict(row, source_id=source_key(row), source_ref=f"모의 API /documents · {source_key(row)}")
    for row in api_rows
]
raw = pd.DataFrame(from_folder + from_api)
display(raw[["source_id", "source_ref", "approval_status", "access_status"]])

taxonomy = {
    "DESIGN": "설계", "CHANGE": "설계", "QVR": "시험·검증",
    "TEST": "시험·검증", "CER": "임상평가", "SUP": "제조·공급망",
    "DMR": "제조·공급망", "STED": "인허가 제출·보완",
}
schema = pd.DataFrame([
    ("document_id + revision", "문자열 복합키", "이 예시의 개정본 식별"),
    ("product_id + project_id + target_market", "문자열", "제출 프로젝트의 제품·시장 범위"),
    ("document_type", "허용값", "Taxonomy에서 매핑"),
    ("regulation_refs", "배열", "확인한 규정과 판본 목록"),
    ("regulation_status", "상태", "판본 확인 여부"),
    ("approval_status", "허용값", "문서 승인 또는 검토 중"),
    ("submission_status", "허용값", "프로젝트별 지정 또는 미지정"),
    ("access_status", "허용값", "원본 접근 가능 또는 접근 제한"),
    ("source_id + source_ref", "문자열", "소스 내 식별자와 위치"),
    ("related_ids", "배열", "관련 요구사항·시험 식별자"),
], columns=["필드", "유형", "규칙"])
raw["document_type"] = raw["source_type"].map(taxonomy).fillna("분류 확인 필요")
assert raw["revision"].str.fullmatch(r"[0-9]+[.][0-9]+").all()
assert not raw["document_type"].eq("분류 확인 필요").any()
display(schema)

scope = ["product_id", "project_id", "target_market"]
key = scope + ["document_id", "revision"]
tracked = ["approval_status", "submission_status", "access_status", "document_type"]
catalog_rows = []
for _, group in raw.groupby(key, sort=False):
    item = group.iloc[0].to_dict()
    item["source_ref"] = " / ".join(sorted(set(group["source_ref"])))
    item["source_count"] = group["source_ref"].nunique()
    item["conflict"] = any(group[field].nunique() > 1 for field in tracked)
    item["revision_order"] = tuple(map(int, item["revision"].split(".")))
    catalog_rows.append(item)
catalog = pd.DataFrame(catalog_rows)
assert len(raw) == 12 and len(catalog) == 11
display(catalog[["document_id", "revision", "document_type", "approval_status",
                 "submission_status", "source_count", "source_ref"]])

,source_id,source_ref,approval_status,access_status
0,CER-001@1.0,폴더 Product-A/CER-001@1.0.json,승인,접근 가능
1,CER-001@1.1,폴더 Product-A/CER-001@1.1.json,검토 중,접근 가능
2,DHF-001@1.0,폴더 Product-A/DHF-001@1.0.json,승인,접근 가능
3,QVR-501@1.0,폴더 Product-A/QVR-501@1.0.json,승인,접근 가능
4,QVR-501@1.1,폴더 Product-A/QVR-501@1.1.json,검토 중,접근 가능
5,STED-001@0.2,폴더 Product-A/STED-001@0.2.json,검토 중,접근 가능
6,QVR-501@1.0,모의 API /documents · QVR-501@1.0,승인,접근 가능
7,DMR-001@1.0,모의 API /documents · DMR-001@1.0,승인,접근 제한
8,TEST-001@2.0,모의 API /documents · TEST-001@2.0,승인,접근 가능
9,QVR-102@1.9,모의 API /documents · QVR-102@1.9,승인,접근 가능


,필드,유형,규칙
0,document_id + revision,문자열 복합키,이 예시의 개정본 식별
1,product_id + project_id + target_market,문자열,제출 프로젝트의 제품·시장 범위
2,document_type,허용값,Taxonomy에서 매핑
3,regulation_refs,배열,확인한 규정과 판본 목록
4,regulation_status,상태,판본 확인 여부
5,approval_status,허용값,문서 승인 또는 검토 중
6,submission_status,허용값,프로젝트별 지정 또는 미지정
7,access_status,허용값,원본 접근 가능 또는 접근 제한
8,source_id + source_ref,문자열,소스 내 식별자와 위치
9,related_ids,배열,관련 요구사항·시험 식별자


,document_id,revision,document_type,approval_status,submission_status,source_count,source_ref
0,CER-001,1.0,임상평가,승인,지정,1,폴더 Product-A/CER-001@1.0.json
1,CER-001,1.1,임상평가,검토 중,미지정,1,폴더 Product-A/CER-001@1.1.json
2,DHF-001,1.0,설계,승인,지정,1,폴더 Product-A/DHF-001@1.0.json
3,QVR-501,1.0,시험·검증,승인,미지정,2,모의 API /documents · QVR-501@1.0 / 폴더 Product-A...
4,QVR-501,1.1,시험·검증,검토 중,지정,1,폴더 Product-A/QVR-501@1.1.json
5,STED-001,0.2,인허가 제출·보완,검토 중,지정,1,폴더 Product-A/STED-001@0.2.json
6,DMR-001,1.0,제조·공급망,승인,지정,1,모의 API /documents · DMR-001@1.0
7,TEST-001,2.0,시험·검증,승인,미지정,1,모의 API /documents · TEST-001@2.0
8,QVR-102,1.9,시험·검증,승인,지정,1,모의 API /documents · QVR-102@1.9
9,QVR-102,1.10,시험·검증,검토 중,미지정,1,모의 API /documents · QVR-102@1.10


## 2. 조회 Tool 직접 실행 · 25분
**함수**는 이름으로 호출할 수 있는 프로그램 조각입니다. `get_document_versions("QVR-501")`에서 따옴표 안의 값은 조회할 문서 번호입니다.  
**바꿔 보기**: 셀 마지막의 문서 번호를 `CER-001`, `UNKNOWN`으로 바꿔 보세요.  
**기대 결과**: QVR-501 v1.0은 승인·미지정, v1.1은 검토 중·지정입니다. 없는 번호는 미확인을 반환합니다.

In [3]:
active_catalog = catalog

def get_document_versions(document_id: str) -> dict:
    '''Product-A(시장 A)의 문서 번호로 개정본, 승인, 제출 지정 및 원본 위치를 조회합니다.'''
    current = active_catalog
    rows = current[current["document_id"].eq(document_id.strip().upper())]
    columns = ["revision", "approval_status", "submission_status", "source_ref"]
    return {
        "document_id": document_id,
        "status": "조회 완료" if len(rows) else "미확인",
        "versions": rows[columns].to_dict("records"),
    }

DOCUMENT_ID = "QVR-501"  # @param {type:"string"}
document_lookup = get_document_versions(DOCUMENT_ID)
display(document_lookup)

{'document_id': 'QVR-501',
 'status': '조회 완료',
 'versions': [{'revision': '1.0',
   'approval_status': '승인',
   'submission_status': '미지정',
   'source_ref': '모의 API /documents · QVR-501@1.0 / 폴더 Product-A/QVR-501@1.0.json'},
  {'revision': '1.1',
   'approval_status': '검토 중',
   'submission_status': '지정',
   'source_ref': '폴더 Product-A/QVR-501@1.1.json'}]}

## 3. 첫 Deep Agent 실행 · 35분

기본 모드는 **도구의 실제 반환값**을 출력해 답변에 넣을 근거를 보여줍니다.  
실제 Agent를 실행하려면 다음 순서로 진행하세요.

1. Colab 왼쪽 열쇠 아이콘(**보안 비밀 / Secrets**)에서 `OPENAI_API_KEY`를 추가하고 이 노트북의 액세스를 켭니다.
2. 아래 `RUN_LIVE_AGENT`를 체크하고 실행합니다. 처음 실행할 때만 패키지 설치가 필요합니다.
3. 사용 권한이 있는 도구 호출 지원 모델을 `MODEL`에 지정합니다. 기본값은 LangChain 공식 예시의 OpenAI 모델입니다.
4. 출력의 **도구 호출 기록**에서 에이전트가 실제로 호출한 이름과 인수를 확인합니다. 실제 호출에는 모델 제공자의 API 사용량이 발생할 수 있습니다.

[LangChain Deep Agents Python Quickstart](https://docs.langchain.com/oss/python/deepagents/quickstart)

In [5]:
RUN_LIVE_AGENT = True  # @param {type:"boolean"}
MODEL = "openai:gpt-6.1-sol"  # @param {type:"string"}

if RUN_LIVE_AGENT:
    import os
    import subprocess
    import sys
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    if not os.environ["OPENAI_API_KEY"]:
        raise ValueError("Colab 보안 비밀에 OPENAI_API_KEY를 추가하세요.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "deepagents", "langchain-openai"
    ])
    from deepagents import create_deep_agent

SYSTEM_PROMPT = (
    "당신은 가상 Product-A 인허가 준비의 RA 업무 보조자입니다. "
    "질문에 필요한 등록 도구를 먼저 호출하세요. "
    "개정본, 승인 및 제출 지정 상태와 원본 위치를 근거로 제시하세요. "
    "제공된 프로그램 판정값을 그대로 사용하세요. "
    "자료가 부족하면 확인 필요라고 표시하고, 승인이나 제출을 완료했다고 표현하지 마세요."
)

def ask_agent(agent, question):
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    calls = [
        {"도구": call["name"], "입력": call["args"]}
        for msg in response["messages"]
        for call in getattr(msg, "tool_calls", [])
    ]
    print("도구 호출 기록")
    display(calls)
    print("Agent 답변")
    display(response["messages"][-1].content)
    return calls

if RUN_LIVE_AGENT:
    first_agent = create_deep_agent(
        model=MODEL, tools=[get_document_versions], system_prompt=SYSTEM_PROMPT
    )
    first_calls = ask_agent(first_agent, f"{DOCUMENT_ID}의 개정본별 승인과 제출 지정 상태는?")
    if not any(call["도구"] == "get_document_versions" for call in first_calls):
        print("조회 도구 호출을 확인하지 못했습니다. 질문에 문서 번호를 명시해 다시 실행하세요.")
else:
    print("기본 모드 · 2단계 도구 결과로 답변에 필요한 근거를 확인하세요.")
    display(document_lookup["versions"])

도구 호출 기록


[{'도구': 'get_document_versions', '입력': {'document_id': 'QVR-501'}}]

Agent 답변


[{'type': 'text',
  'text': 'QVR-501의 개정본별 승인 및 제출 지정 상태는 다음과 같습니다.\n\n| 개정본 | 승인 상태 | 제출 지정 상태 | 원본 위치 |\n|---|---|---|---|\n| 1.0 | 승인 | 미지정 | 모의 API `/documents` · `QVR-501@1.0` / 폴더 `Product-A/QVR-501@1.0.json` |\n| 1.1 | 검토 중 | 지정 | 폴더 `Product-A/QVR-501@1.1.json` |\n\n※ **제출 지정은 제출 완료를 의미하지 않습니다.** 실제 제출 완료 여부는 확인이 필요합니다.',
  'annotations': [],
  'id': 'msg_0072f7c37803c155006afc03aa6fec87d18c78b604d3b29044',
  'phase': 'final_answer'}]

## 4. 프로젝트 제출 준비 판정 · 40분
**관찰**: 문서 한 건이 준비되면 프로젝트 전체도 준비될까요?

- 다음 셀의 값이 `False`이면 기준 결과는 **3/8건 준비**입니다.
- `True`로 바꾸고 다시 실행하면 가상으로 QVR-501 v1.1 승인이 반영되어 **4/8건 준비**입니다.
- 두 경우 모두 프로젝트 판정은 **실습 제출 조건 미충족**입니다.

준비 조건은 제출 지정 개정본 1건, 문서 유형 일치, 원본 접근 가능, 승인입니다. 복수 지정이나 출처 충돌도 확인 대상으로 처리합니다. 모델 연결 여부와 무관하게 프로그램이 동일한 규칙으로 계산합니다.

In [6]:
def summarize_versions(group):
    ordered = group.sort_values("revision_order", ascending=False)
    approved = ordered[ordered["approval_status"].eq("승인") & ~ordered["conflict"]]
    selected = ordered[ordered["submission_status"].eq("지정")]
    return {
        "최신 작성본": ordered.iloc[0]["revision"],
        "최신 승인본": approved.iloc[0]["revision"] if len(approved) else "—",
        "제출 지정본": ", ".join(selected["revision"]) if len(selected) else "—",
    }

version_table = pd.DataFrame([
    dict(document_id=doc_id, **summarize_versions(group))
    for doc_id, group in catalog.groupby("document_id")
])
display(version_table)

,document_id,최신 작성본,최신 승인본,제출 지정본
0,CER-001,1.1,1.0,1.0
1,CR-001,1.0,1.0,—
2,DHF-001,1.0,1.0,1.0
3,DMR-001,1.0,1.0,1.0
4,QVR-102,1.10,1.9,1.9
5,QVR-501,1.1,1.0,1.1
6,STED-001,0.2,—,0.2
7,TEST-001,2.0,2.0,—


In [7]:
APPROVE_QVR_501 = True  # @param {type:"boolean"}

working = catalog.copy()
if APPROVE_QVR_501:
    revised = working["document_id"].eq("QVR-501") & working["revision"].eq("1.1")
    working.loc[revised, "approval_status"] = "승인"

active_catalog = working

def assess(group, expected_type):
    if group.empty:
        return "누락", "원본 위치와 확보 일정을 확인"
    if group["conflict"].any():
        return "출처 간 충돌", "원본 시스템에서 메타데이터를 대조"
    designated = group[group["submission_status"].eq("지정")]
    if len(designated) != 1:
        return "제출 지정 확인", "RA와 제출 지정본을 하나로 확정"
    row = designated.iloc[0]
    if row["document_type"] != expected_type:
        return "유형 확인", "필수 목록과 문서 분류를 대조"
    if row["access_status"] != "접근 가능":
        return "접근 제한", "권한 또는 승인된 공유 경로를 확인"
    if row["approval_status"] != "승인":
        return "승인 대기", "지정 개정본의 검토 및 승인 근거를 확인"
    return "준비", "제출 검토에 전달"

scoped = working[(working[scope] == [PRODUCT, PROJECT, MARKET]).all(axis=1)]
results = []
for requirement in required.to_dict("records"):
    docs = scoped[scoped["document_id"].eq(requirement["document_id"])]
    status, action = assess(docs, requirement["예상 분류"])
    versions = summarize_versions(docs) if len(docs) else dict.fromkeys(
        ["최신 작성본", "최신 승인본", "제출 지정본"], "—"
    )
    results.append(dict(requirement, **versions, 상태=status, 다음_조치=action,
                        관찰_근거=" / ".join(sorted(set(docs["source_ref"]))) if len(docs) else "두 소스 목록에서 미확인"))
readiness = pd.DataFrame(results)
bottlenecks = readiness[readiness["상태"].ne("준비")].copy()
print(f"준비 문서 {len(readiness) - len(bottlenecks)}/{len(readiness)}건")
print("실습 제출 조건 충족" if bottlenecks.empty else "실습 제출 조건 미충족")
display(bottlenecks[["필수 문서", "제출 지정본", "상태", "책임부서",
                     "영향 단계", "관찰_근거", "다음_조치"]])

준비 문서 4/8건
실습 제출 조건 미충족


,필수 문서,제출 지정본,상태,책임부서,영향 단계,관찰_근거,다음_조치
3,협력업체 승인원,—,누락,구매,D,두 소스 목록에서 미확인,원본 위치와 확보 일정을 확인
4,제조 기준 문서,1.0,접근 제한,제조,D,모의 API /documents · DMR-001@1.0,권한 또는 승인된 공유 경로를 확인
5,기술문서 제출본,0.2,승인 대기,RA,E,폴더 Product-A/STED-001@0.2.json,지정 개정본의 검토 및 승인 근거를 확인
6,인증시험 성적서,—,제출 지정 확인,품질,E,모의 API /documents · TEST-001@2.0,RA와 제출 지정본을 하나로 확정


In [8]:
def check_submission_readiness() -> dict:
    '''Product-A 필수 문서의 준비 건수 및 차단 사유와 담당부서를 조회합니다.'''
    return {
        "프로젝트": PROJECT,
        "대상_시장": MARKET,
        "준비": len(readiness) - len(bottlenecks),
        "전체": len(readiness),
        "판정": "실습 제출 조건 충족" if bottlenecks.empty else "실습 제출 조건 미충족",
        "차단_항목": bottlenecks.to_dict("records"),
    }

official_readiness = check_submission_readiness()
print(official_readiness["판정"])
display(bottlenecks[["document_id", "상태", "책임부서", "다음_조치"]])

실습 제출 조건 미충족


,document_id,상태,책임부서,다음_조치
3,SUP-001,누락,구매,원본 위치와 확보 일정을 확인
4,DMR-001,접근 제한,제조,권한 또는 승인된 공유 경로를 확인
5,STED-001,승인 대기,RA,지정 개정본의 검토 및 승인 근거를 확인
6,TEST-001,제출 지정 확인,품질,RA와 제출 지정본을 하나로 확정


## 5. 보완 요청 대응 초안 · 35분
가상 기관 요청은 **“거리 측정 시험 조건과 결과를 보완해 주세요”**입니다. 강사가 제공한 미니 추적표에서 이 요청을 `EX-SRS-501`과 연결합니다. 시험 근거 연결 도구가 어떤 자료를 찾는지 먼저 보고, 실제 Agent 모드에서는 등록 도구를 호출해 RA 검토용 답변을 작성합니다.

카탈로그에는 문서의 위치와 상태가 들어 있습니다. 시험 조건이나 실측값이 조회되지 않으면 **원문 확인 필요**로 남깁니다.

In [9]:
TRACE_MATRIX = {
    "EX-SRS-501": {
        "요구사항": "정지 영상에서 두 점 간 거리를 표시한다.",
        "검증_계획": "EX-QVS-501",
        "시험_문서": "QVR-501",
        "담당부서": "품질",
        "출처": "교육용 Traceability Matrix 예시",
    }
}

def find_requirement_evidence(requirement_id: str) -> dict:
    '''교육용 추적표에서 요구사항에 연결된 시험 문서와 개정본별 근거를 조회합니다.'''
    requirement_id = requirement_id.strip().upper()
    link = TRACE_MATRIX.get(requirement_id)
    if link is None:
        return {"requirement_id": requirement_id, "status": "확인 필요"}
    versions = get_document_versions(link["시험_문서"])
    return {"requirement_id": requirement_id, **link, "개정본_조회": versions,
            "추가_확인": "시험 조건과 실측 결과는 시험 원문 확인 필요"}

evidence = find_requirement_evidence("EX-SRS-501")
supplement_request = "거리 측정 시험 조건과 결과를 보완해 주세요."
supplement_draft = pd.DataFrame([{
    "보완 요청": supplement_request,
    "요구사항 ID": evidence["requirement_id"],
    "대응 근거 후보": evidence["시험_문서"],
    "제출 지정 개정본": readiness.set_index("document_id").loc["QVR-501", "제출 지정본"],
    "담당부서": evidence["담당부서"],
    "추가 확인": evidence["추가_확인"],
    "RA 확인": "검토 대기",
}])
display(supplement_draft)

,보완 요청,요구사항 ID,대응 근거 후보,제출 지정 개정본,담당부서,추가 확인,RA 확인
0,거리 측정 시험 조건과 결과를 보완해 주세요.,EX-SRS-501,QVR-501,1.1,품질,시험 조건과 실측 결과는 시험 원문 확인 필요,검토 대기


In [10]:
if RUN_LIVE_AGENT:
    workflow_agent = create_deep_agent(
        model=MODEL,
        tools=[get_document_versions, check_submission_readiness, find_requirement_evidence],
        system_prompt=SYSTEM_PROMPT,
    )
    workflow_calls = ask_agent(workflow_agent, (
        f"검증 프로그램의 판정은 {official_readiness['판정']}이고 "
        f"준비 문서는 {official_readiness['준비']}/{official_readiness['전체']}건입니다. "
        "check_submission_readiness를 호출해 남은 병목을 요약하세요. "
        f"기관 보완 요청은 '{supplement_request}'입니다. "
        "EX-SRS-501의 시험 근거를 도구로 확인하고 RA 검토용 대응표 초안을 작성하세요. "
        "개정본 상태가 필요하면 승인 시나리오가 반영된 준비 판정 결과와 대조하세요."
    ))
else:
    print("기본 모드 · 위 표가 프로그램이 만든 RA 검토용 보완 대응 초안입니다.")

도구 호출 기록


[{'도구': 'check_submission_readiness', '입력': {}},
 {'도구': 'find_requirement_evidence', '입력': {'requirement_id': 'EX-SRS-501'}},
 {'도구': 'get_document_versions', '입력': {'document_id': 'QVR-501'}}]

Agent 답변


[{'type': 'text',
  'text': '## 1. 준비 판정 및 남은 병목\n\n`check_submission_readiness` 조회 결과, Product-A(시장 A)의 프로그램 판정은 **실습 제출 조건 미충족**, 준비 문서는 **4/8건**입니다.\n\n| 차단 문서 / 책임부서 | 최신 작성본 / 승인본 / 제출 지정본 | 병목 및 다음 조치 | 조회 근거·원본 위치 |\n|---|---|---|---|\n| SUP-001 협력업체 승인원 / 구매 | — / — / — | **누락**: 원본 위치와 확보 일정 확인 필요 | 두 소스 목록에서 미확인. 원본 위치 확인 필요 |\n| DMR-001 제조 기준 문서 / 제조 | 1.0 / 1.0 / 1.0 | **접근 제한**: 열람 권한 또는 승인된 공유 경로 확인 필요 | 모의 API `/documents` · `DMR-001@1.0` |\n| STED-001 기술문서 제출본 / RA | 0.2 / — / 0.2 | **승인 대기**: 지정 개정본의 검토·승인 근거 확인 필요 | `Product-A/STED-001@0.2.json` |\n| TEST-001 인증시험 성적서 / 품질 | 2.0 / 2.0 / — | **제출 지정 확인**: RA와 제출 지정본을 하나로 확정 필요 | 모의 API `/documents` · `TEST-001@2.0` |\n\n승인본 또는 제출 지정본이 있는 문서도 접근 제한 등 별도 차단 사유가 남아 있습니다. 따라서 이를 근거로 준비 판정을 변경하지 않습니다.\n\n## 2. EX-SRS-501 시험 근거 확인\n\n- **요구사항:** 정지 영상에서 두 점 간 거리를 표시한다.\n- **검증 계획:** EX-QVS-501\n- **연결 시험 문서:** QVR-501\n- **담당부서:** 품질\n- **추적 관계 출처:** 교육용 Traceability Matrix 예시\n\n`find_requirement_evidence`의 개정본 조회 결과와 `get_

## 6. 팀별 검증 및 발표 · 25분

1. 기준 상태에서 `QVR-501`(승인 대기), `SUP-001`(누락), `DMR-001`(접근 제한)을 구분합니다.
2. 가상 승인을 적용한 뒤 QVR-501 차단이 해소되고 다른 차단은 남는지 설명합니다.
3. 실제 Agent 모드를 켰다면 도구 호출 기록, Tool 결과, Agent 답변을 대조합니다. Agent가 추정한 시험 조건이나 수치가 있다면 원문 확인 대상으로 표시합니다.
4. 결과를 내보낸 뒤 Colab 왼쪽 **파일** 탭에서 `product_a_deepagents_results.zip`을 다운로드합니다.

In [11]:
status_by_id = readiness.set_index("document_id")["상태"]
assert status_by_id["QVR-501"] == ("준비" if APPROVE_QVR_501 else "승인 대기")
assert status_by_id["SUP-001"] == "누락"
assert status_by_id["DMR-001"] == "접근 제한"
assert status_by_id["TEST-001"] == "제출 지정 확인"
assert status_by_id["STED-001"] == "승인 대기"
assert status_by_id["CER-001"] == "준비"
assert version_table.set_index("document_id").loc["QVR-102", "최신 작성본"] == "1.10"
assert len(bottlenecks) == (4 if APPROVE_QVR_501 else 5)
print("기준 예제와 대조 완료")

기준 예제와 대조 완료


In [12]:
with ZipFile("product_a_deepagents_results.zip", "w") as archive:
    for name, table in {
        "submission_readiness": readiness,
        "bottlenecks": bottlenecks,
        "supplement_draft": supplement_draft,
    }.items():
        archive.writestr(f"{name}.csv", table.to_csv(index=False).encode("utf-8-sig"))
print("내보내기 완료 · product_a_deepagents_results.zip")

내보내기 완료 · product_a_deepagents_results.zip
